# Synthetic delusional chat - thin driver

All logic lives in `src/` now: chat generation (prompts, planning, refusal handling,
checkpointed driver, bucket verification) in `src/lib/chatgen.py`; the annotation machinery
(settings, store, batch driver) shared with the WildChat pipeline via `src/lib/annotation.py`
+ `src/lib/store.py`, unpacked from `config.toml` in both main files. CLI equivalents:
`python src/synthetic.py status | generate | annotate | verify`.

Per theme: 2,000 annotated messages (user side + bot side both count), 1,000 positive /
1,000 negative buckets + `margin` (see `synthetic.toml`). Re-running `generate`/`annotate`
tops up only what is missing; `verify` reports realised buckets from actual annotator scores.


In [ ]:
import sys
from pathlib import Path

# make `src.*` imports work no matter where the kernel was started
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
sys.path.insert(0, str(ROOT))

import src.synthetic as syn

print("generator   :", syn.SETTINGS.model, "| themes:", syn.SETTINGS.themes)
print("chats file  :", syn.CHATS_FILE.relative_to(syn.PROJECT_ROOT))
print("shard store :", syn.SHARD_DIR.relative_to(syn.PROJECT_ROOT))
print("annotator   : shared via src/lib (annotation model -", syn.ANNOTATION_SETTINGS.model + ")")


In [ ]:
# pipeline status: instructed buckets vs target+margin, store pending counts
syn.status()


In [ ]:
# stage 1/2: generate scripted conversations (3-6 turns) via Ollama, one message per call
# (CLI: python src/synthetic.py generate)
syn.run_generation()


In [ ]:
# peek at the most recent conversation ("pos"/"neg" = the scripted target per message)
syn.peek()


In [ ]:
# stage 2/2: annotate staged messages not yet stored, into data/synthetic-annotations/shards
# (CLI: python src/synthetic.py annotate)
store = syn.annotate()
annotated = store.dataset()
annotated


In [ ]:
# verify: realised positive/negative buckets per theme (score >= packaged cutoff)
# (CLI: python src/synthetic.py verify)
syn.verify(store)
